# Anal?za z?znam? robot?
budeme pracovat s daty robotu z nasich databazi. Podivame se na nastroje `BigData`

## Import knihoven a konstanty

In [ ]:
import os
import datetime
import pandas
import sqlalchemy as sql

In [ ]:
__file__ = __vsc_ipynb_file__  
SCRIPT_FOLDER = os.path.dirname(__file__)
PROJECT_FOLDER = os.path.dirname(SCRIPT_FOLDER)
DATA_FOLDER = os.path.join(PROJECT_FOLDER,'data')
WORK_FOLDER = os.path.join(DATA_FOLDER,'working')

COL_TIME = 'cas_robota'
COL_ROBOT = 'robot'
COL_DAY = 'day'
COL_MINUTE = 'minute'

PRQT = os.path.join(WORK_FOLDER,'roboti.prqt')
CSV = os.path.join(WORK_FOLDER,'roboti.csv')
XLS = os.path.join(WORK_FOLDER,'roboti.xlsx')
PRQT_PART = os.path.join(WORK_FOLDER,'roboti.part')

## P??stupov? ?daje do datab?z?

In [ ]:
DBSURL = "https://development.techniarch.com/pcsda/robot.php?dbs=psycopg2"
DBUSER = 'robot'
DBPASS = 'Kurz-DAPYT_ST45'

## Z?sk?n? seznamu datab?z?

In [ ]:
dbs = pandas.read_csv(DBSURL)
dbs

## P?e?ten? dat robot?
z SQL DB a vytvo?en? spole?n?ho `DataFrame`

In [ ]:
dfs = []
start = pandas.Timestamp.now()
for idx,row in dbs.iterrows():
    start_sql = pandas.Timestamp.now()
    try:    
        dbcon = sql.create_engine(row['conn_string'].format(dbuser=DBUSER,dbpwd=DBPASS))
        df = pandas.read_sql_table('cesta_robota',dbcon)
        df[COL_ROBOT] = row['conn_name']
        dfs.append(df)
        print(f"z DB `{row['conn_name']}` jsme preccetli {df.shape[0]} radku za {(pandas.Timestamp.now() - start_sql).total_seconds()} sekund")
    except Exception as err:
        print(f"Chyba pri cteni databaze `{row['conn_name']}`:", err)
robots = pandas.concat(dfs)
print(f'z databazi jsme precetli {robots.shape[0]} radku za {(pandas.Timestamp.now() - start).total_seconds()} sekund')
robots.info()
robots

## Drobn? ?pravy v robotech

In [ ]:
robots[COL_DAY] = robots[COL_TIME].dt.date
robots[COL_MINUTE] = robots[COL_TIME].dt.floor('min')
robots.info()
robots

In [ ]:
robots.set_index([COL_ROBOT,COL_MINUTE],inplace=True)
robots.info()
robots

In [ ]:
robots.loc['tomask']

## Ulo?en? DF na lok?ln? disk
### CSV

In [ ]:
start = pandas.Timestamp.now()
robots.to_csv(CSV)
print(f"roboty jsme zapsali do {CSV} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")


### Excel

In [ ]:
start = pandas.Timestamp.now()
robots.to_excel(XLS)
print(f"roboty jsme zapsali do {XLS} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

### PARQUET
Soubory `parquet`
- bin?rn? soubor(y)
- soubory jsou komprimovan?
- umo??uj? indexov?n? a partitioning
- nen? mo?n? soubor m?nit, jen p?epsat /POZOR: na adres??ov? struktury/
- p?ipraveno pro velk? objemy data - `BigData`

In [ ]:
start = pandas.Timestamp.now()
robots.to_parquet(PRQT)
print(f"roboty jsme zapsali do {PRQT} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

## ?ten? cel?ho souboru
### CSV

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_csv(CSV)
print(f"roboty mame z csv {CSV} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

### PARQUET

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_parquet(PRQT)
print(f"roboty mame z csv {PRQT} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

## ?ten? ??sti dat
(pot?ebujeme data jen jednoho robota)

### Klasick? p??stup
p?e?teme v?e a filtrujem

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_csv(CSV,index_col=[COL_ROBOT,COL_DAY])
vcera = df.loc[('tomask','2025-11-02')]
print(f"roboty mame z csv {CSV} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")
print(f"precteno celkem {df.shape[0]} radku z toho za vcerejsek {vcera.shape[0]}")

#### Parquet

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_parquet(PRQT)
vcera = df[df[COL_DAY]==datetime.date(2025,11,2)].loc['tomask']
print(f"roboty mame z csv {PRQT} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")
print(f"precteno celkem {df.shape[0]} radku z toho za vcerejsek {vcera.shape[0]}")

### Vyu?it? PARQUET partitioning

#### 1. ulo??me data rozd?len? na ??sti `partition`

In [ ]:
start = pandas.Timestamp.now()
robots.to_parquet(PRQT_PART,partition_cols=[COL_ROBOT,COL_DAY])
print(f"roboty jsme zapsali do {PRQT_PART} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

#### 2. ?teme data s vyu?it?m filtru

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_parquet(PRQT_PART,filters=[[(COL_ROBOT,'==','tomask'),(COL_DAY,'==','2025-11-02')]],columns=['kostka1','kostka2'])
#vcera = df[df[COL_DAY]==datetime.date(2025,11,2)].loc['tomask']
print(f"roboty mame z csv {PRQT_PART} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")
print(f"precteno celkem {df.shape[0]} radku")
df

In [ ]:
start = pandas.Timestamp.now()
df =  pandas.read_parquet(PRQT,columns=['kostka1','kostka2'])
print(f"roboty mame z csv {PRQT} za {(pandas.Timestamp.now() - start).total_seconds()} sekund")

In [ ]:
df